# Web y APIs · Peticiones y *web scraping*

Peticiones HTTP con `requests` y extracción de datos de páginas web con `BeautifulSoup` y `pandas`. Necesita conexión a internet.

## 1. Petición y respuesta HTTP

Hacer una petición `GET` a una API pública (la de GitHub) y estudiar la respuesta: código de estado, cabeceras y cuerpo en JSON.

In [1]:
import requests

CABECERAS = {"User-Agent": "logic-programming-exercises/1.0"}
respuesta = requests.get("https://api.github.com/users/Asabeneh", headers=CABECERAS, timeout=30)
print(respuesta.status_code, respuesta.reason, respuesta.headers["Content-Type"])
datos = respuesta.json()
print({k: datos[k] for k in ("login", "name", "public_repos")})
print(requests.get("https://api.github.com/users/no-existe-este-usuario-12345", headers=CABECERAS, timeout=30).status_code)

200 OK application/json; charset=utf-8
{'login': 'Asabeneh', 'name': 'Asabeneh', 'public_repos': 197}
404


## 2. Datos de una tabla de Wikipedia

Extraer la tabla de información (*infobox*) del artículo de Python en Wikipedia y crear una función que obtenga todos los enlaces de la sección de referencias.

In [2]:
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Python_(programming_language)"
soup = BeautifulSoup(requests.get(url, headers=CABECERAS, timeout=30).text, "lxml")
infobox = soup.find("table", class_="infobox")
tabla = {fila.th.get_text(strip=True): fila.td.get_text(strip=True) for fila in infobox.find_all("tr") if fila.th and fila.td}
print(tabla["Developer"])

def enlaces_referencias(soup):
    # la lista de referencias va tras el encabezado con id "References"
    referencias = soup.find(id="References").find_next("ol", class_="references")
    return [a["href"] for a in referencias.select("a.external")]

enlaces = enlaces_referencias(soup)
print(len(enlaces), enlaces[:3])

Python Software Foundation
375 ['https://docs.python.org/3/faq/general.html#what-is-python', 'https://www.tuhs.org/Usenet/alt.sources/1991-February/001749.html', 'https://web.archive.org/web/20210811171015/https://www.tuhs.org/Usenet/alt.sources/1991-February/001749.html']


## 3. Datos de la Universidad de Boston a JSON

Extraer las cifras de *BU Facts & Stats* agrupadas por sección y guardarlas en un fichero JSON.

In [3]:
import json

pagina = requests.get("https://www.bu.edu/president/boston-university-facts-stats/", headers=CABECERAS, timeout=30)
soup = BeautifulSoup(pagina.text, "lxml")
datos = {}
for seccion in soup.select("section.stat-section"):
    titulo = seccion.select_one(".stat-group-title").get_text(strip=True)
    datos[titulo] = {li.select_one(".stat-label").get_text(strip=True): li.select_one(".stat-figure").get_text(strip=True)
                     for li in seccion.select("li")}
with open("bu_facts.json", "w", encoding="utf-8") as f:
    json.dump(datos, f, ensure_ascii=False, indent=2)
print(json.dumps(datos, ensure_ascii=False)[:300])

{"Community": {"Student Body": "38,093", "Living Alumni": "460,000+", "Total Employees": "11,532", "Faculty": "4,534", "Nondegree Students": "1,499", "Graduate & Professional Students": "18,341", "Undergraduate Students": "18,253"}, "Academics": {"Study Abroad Programs": "170+", "Average Class Size"


## 4. Conjuntos de datos de la UCI a JSON

La tabla de `archive.ics.uci.edu/ml/datasets.php` ya no existe (el sitio nuevo se genera con JavaScript), pero el repositorio ofrece una API. Guardar la lista de conjuntos de datos en JSON.

In [4]:
uci = requests.get("https://archive.ics.uci.edu/api/datasets/list", headers=CABECERAS, timeout=30).json()["data"]
with open("uci_datasets.json", "w", encoding="utf-8") as f:
    json.dump(uci, f, ensure_ascii=False, indent=2)
print(len(uci), uci[:3])

689 [{'id': 1, 'name': 'Abalone'}, {'id': 2, 'name': 'Adult'}, {'id': 3, 'name': 'Annealing'}]


## 5. Presidentes de Estados Unidos a JSON

Extraer la tabla de presidentes de Wikipedia y guardarla en JSON. `pandas.read_html` convierte las tablas HTML en DataFrames.

In [5]:
from io import StringIO
import pandas as pd

html = requests.get("https://en.wikipedia.org/wiki/List_of_presidents_of_the_United_States", headers=CABECERAS, timeout=30).text
tablas = pd.read_html(StringIO(html))
presidentes = next(t for t in tablas if any("Name" in str(c) for c in t.columns))
presidentes.columns = [" ".join(dict.fromkeys(map(str, c))) if isinstance(c, tuple) else str(c) for c in presidentes.columns]
presidentes.columns = presidentes.columns.str.replace(r"\[.*?\]", "", regex=True)
presidentes = presidentes.loc[:, ~presidentes.columns.duplicated()]
# las celdas combinadas repiten filas: una por presidente
presidentes = presidentes.drop_duplicates(subset=presidentes.columns[0])
presidentes.to_json("presidentes.json", orient="records", force_ascii=False, indent=2)
print(presidentes.shape)
print(list(presidentes.columns))
print(presidentes.iloc[:5, [0, 2]].to_string(index=False))

(47, 8)
['No.', 'Portrait', 'Name (birth–death)', 'Term', 'Party', 'Party.1', 'Election', 'Vice President']
 No.                 Name (birth–death)
   1 George Washington (1732–1799) [19]
   2        John Adams (1735–1826) [21]
   3  Thomas Jefferson (1743–1826) [23]
   4     James Madison (1751–1836) [24]
   5      James Monroe (1758–1831) [26]
